In [1]:
from pathlib import Path

import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    make_scorer,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from src.utils.features import split_features

In [2]:
DATA_DIR = Path("../datasets/final/ml")

DATASET_NAMES = [
    "cicids2017",
    "unsw_nb15",
    "iot23",
]

RANDOM_STATE = 1
N_SPLITS = 5

In [3]:
def build_logistic_pipeline(numerical_features, categorical_features):
    preprocessor = ColumnTransformer(
        transformers=[
            ("numerical", StandardScaler(), numerical_features),
            ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ]
    )

    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", LogisticRegression(
                solver="lbfgs",
                max_iter=1000,
                random_state=RANDOM_STATE,
            )),
        ]
    )

In [4]:
def summarize_coefficients(estimators):
    coefficients_by_fold = []

    for fold, estimator in enumerate(estimators, start=1):
        preprocessor = estimator.named_steps["preprocessor"]
        model = estimator.named_steps["model"]

        feature_names = [
            name.removeprefix("numerical__").removeprefix("categorical__")
            for name in preprocessor.get_feature_names_out()
        ]

        coefficients_by_fold.append(pd.DataFrame({
            "fold": fold,
            "feature": feature_names,
            "coefficient": model.coef_[0],
        }))

    coefficients = pd.concat(coefficients_by_fold, ignore_index=True)
    summary = (
        coefficients
        .groupby("feature")["coefficient"]
        .agg(["mean", "std"])
        .rename(columns={
            "mean": "coefficient_mean",
            "std": "coefficient_std",
        })
        .reset_index()
    )

    summary["abs_coefficient_mean"] = summary["coefficient_mean"].abs()

    return summary.sort_values("abs_coefficient_mean", ascending=False)


def evaluate_training_dataset(dataset_name):
    df = pd.read_parquet(
        DATA_DIR / f"{dataset_name}_train.parquet"
    )

    numerical_features, categorical_features = split_features(df)
    feature_columns = numerical_features + categorical_features

    features = df[feature_columns]
    target = df["label_binary"].eq("ATTACK").astype(int)

    cross_validation = StratifiedKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

    scores = cross_validate(
        build_logistic_pipeline(
            numerical_features,
            categorical_features,
        ),
        features,
        target,
        cv=cross_validation,
        scoring={
            "precision_attack": make_scorer(
                precision_score,
                pos_label=1,
                zero_division=0,
            ),
            "recall_attack": make_scorer(
                recall_score,
                pos_label=1,
                zero_division=0,
            ),
            "f1_attack": make_scorer(f1_score, pos_label=1),
            "accuracy": "accuracy",
            "balanced_accuracy": "balanced_accuracy",
        },
        n_jobs=1,
        return_estimator=True,
    )

    metrics = {
        "dataset": dataset_name,
        "numerical_features": len(numerical_features),
        "categorical_features": len(categorical_features),
        "precision_attack_mean": scores["test_precision_attack"].mean(),
        "precision_attack_std": scores["test_precision_attack"].std(),
        "recall_attack_mean": scores["test_recall_attack"].mean(),
        "recall_attack_std": scores["test_recall_attack"].std(),
        "f1_attack_mean": scores["test_f1_attack"].mean(),
        "f1_attack_std": scores["test_f1_attack"].std(),
        "accuracy_mean": scores["test_accuracy"].mean(),
        "accuracy_std": scores["test_accuracy"].std(),
        "balanced_accuracy_mean": scores["test_balanced_accuracy"].mean(),
        "balanced_accuracy_std": scores["test_balanced_accuracy"].std(),
    }

    return metrics, summarize_coefficients(scores["estimator"])

In [5]:
results = []
coefficient_summaries = {}

for dataset_name in DATASET_NAMES:
    metrics, coefficient_summary = evaluate_training_dataset(dataset_name)
    results.append(metrics)
    coefficient_summaries[dataset_name] = coefficient_summary

results_df = pd.DataFrame(results)

display(results_df)

for dataset_name in DATASET_NAMES:
    print(f"\nCoeficientes - {dataset_name}")
    display(coefficient_summaries[dataset_name])

,dataset,numerical_features,categorical_features,precision_attack_mean,precision_attack_std,recall_attack_mean,recall_attack_std,f1_attack_mean,f1_attack_std,accuracy_mean,accuracy_std,balanced_accuracy_mean,balanced_accuracy_std
0,cicids2017,14,1,0.884955,0.003234,0.904433,0.002587,0.894586,0.002646,0.95737,0.001093,0.937519,0.001592
1,unsw_nb15,14,1,0.937062,0.001055,0.818421,0.002909,0.873730,0.001748,0.95269,0.000589,0.902339,0.001442
2,iot23,14,1,0.995054,0.001068,0.990976,0.000991,0.993011,0.000987,0.99721,0.000394,0.994872,0.000611



Coeficientes - cicids2017


,feature,coefficient_mean,coefficient_std,abs_coefficient_mean
14,protocol_17,-5.284778,0.118801,5.284778
9,bidirectional_psh_packets,-4.812710,0.450360,4.812710
12,bidirectional_stddev_ps,4.166747,0.119401,4.166747
5,bidirectional_max_ps,-1.797194,0.077724,1.797194
15,protocol_6,1.731079,0.037179,1.731079
10,bidirectional_rst_packets,1.367040,0.010592,1.367040
3,bidirectional_fin_packets,-1.272407,0.019027,1.272407
7,bidirectional_mean_ps,-0.939773,0.045247,0.939773
4,bidirectional_max_piat_ms,-0.799276,0.024679,0.799276
13,bidirectional_syn_packets,0.791913,0.010045,0.791913



Coeficientes - unsw_nb15


,feature,coefficient_mean,coefficient_std,abs_coefficient_mean
0,bidirectional_ack_packets,52.459797,3.264637,52.459797
9,bidirectional_psh_packets,-46.259402,0.753663,46.259402
8,bidirectional_packets,5.871671,2.617693,5.871671
1,bidirectional_bytes,-5.616012,2.325864,5.616012
12,bidirectional_stddev_ps,-4.849600,0.097347,4.849600
5,bidirectional_max_ps,3.601476,0.083304,3.601476
16,protocol_89,-3.062205,0.284295,3.062205
6,bidirectional_mean_piat_ms,1.416856,0.542857,1.416856
11,bidirectional_stddev_piat_ms,-1.407093,0.573136,1.407093
14,protocol_17,1.039594,0.356058,1.039594



Coeficientes - iot23


,feature,coefficient_mean,coefficient_std,abs_coefficient_mean
6,bidirectional_mean_piat_ms,-15.453220,0.275454,15.453220
14,protocol_17,-8.068413,0.702838,8.068413
8,bidirectional_packets,-6.681116,2.580834,6.681116
0,bidirectional_ack_packets,-6.127290,1.847045,6.127290
9,bidirectional_psh_packets,5.752156,2.897120,5.752156
2,bidirectional_duration_ms,5.079685,0.688021,5.079685
3,bidirectional_fin_packets,4.862230,0.536042,4.862230
4,bidirectional_max_piat_ms,3.731596,1.043608,3.731596
15,protocol_6,3.606490,0.215658,3.606490
1,bidirectional_bytes,-2.728221,0.561635,2.728221


## Validação cross-dataset de protocol

In [6]:
def evaluate_protocol_cross_dataset():
    results = []

    for source_name in DATASET_NAMES:
        source_df = pd.read_parquet(
            DATA_DIR / f"{source_name}_train.parquet"
        )

        numerical_features, categorical_features = split_features(source_df)

        for use_protocol in [True, False]:
            current_categorical = categorical_features.copy()

            if not use_protocol:
                current_categorical.remove("protocol")

            feature_columns = numerical_features + current_categorical
            source_target = (
                source_df["label_binary"].eq("ATTACK").astype(int)
            )

            pipeline = build_logistic_pipeline(
                numerical_features,
                current_categorical,
            )
            pipeline.fit(source_df[feature_columns], source_target)

            for target_name in DATASET_NAMES:
                if target_name == source_name:
                    continue

                target_df = pd.read_parquet(
                    DATA_DIR / f"{target_name}_train.parquet",
                    columns=feature_columns + ["label_binary"],
                )
                target = (
                    target_df["label_binary"].eq("ATTACK").astype(int)
                )
                predictions = pipeline.predict(target_df[feature_columns])

                results.append({
                    "source_dataset": source_name,
                    "validation_dataset": target_name,
                    "protocol": "included" if use_protocol else "excluded",
                    "precision_attack": precision_score(
                        target,
                        predictions,
                        zero_division=0,
                    ),
                    "recall_attack": recall_score(
                        target,
                        predictions,
                        zero_division=0,
                    ),
                    "f1_attack": f1_score(target, predictions),
                    "accuracy": accuracy_score(target, predictions),
                    "balanced_accuracy": balanced_accuracy_score(
                        target,
                        predictions,
                    ),
                })

    return pd.DataFrame(results)

In [7]:
protocol_results = evaluate_protocol_cross_dataset()
display(protocol_results)

metric_columns = [
    "precision_attack",
    "recall_attack",
    "f1_attack",
    "accuracy",
    "balanced_accuracy",
]
index_columns = ["source_dataset", "validation_dataset"]

included = (
    protocol_results[protocol_results["protocol"] == "included"]
    .set_index(index_columns)
)
excluded = (
    protocol_results[protocol_results["protocol"] == "excluded"]
    .set_index(index_columns)
)

# diferença = sem protocol - com protocol
#lembrete: valores negativos indicam piora após a remoção
protocol_impact = excluded[metric_columns] - included[metric_columns]
protocol_impact.columns = [
    f"{column}_difference" for column in metric_columns
]

display(protocol_impact.reset_index())

,source_dataset,validation_dataset,protocol,precision_attack,recall_attack,f1_attack,accuracy,balanced_accuracy
0,cicids2017,unsw_nb15,included,0.091954,0.001699,0.003336,0.796985,0.498753
1,cicids2017,iot23,included,0.028670,0.104803,0.045023,0.110816,0.108561
2,cicids2017,unsw_nb15,excluded,0.088170,0.001677,0.003292,0.796866,0.498670
3,cicids2017,iot23,excluded,0.028620,0.104505,0.044934,0.111504,0.108879
4,unsw_nb15,cicids2017,included,0.599951,0.413882,0.489842,0.827581,0.672444
5,unsw_nb15,iot23,included,0.004456,0.016009,0.006971,0.087813,0.060886
6,unsw_nb15,cicids2017,excluded,0.570841,0.409508,0.476899,0.820328,0.666270
7,unsw_nb15,iot23,excluded,0.004284,0.015393,0.006702,0.087481,0.060448
8,iot23,cicids2017,included,0.404885,0.324833,0.360468,0.769476,0.602735
9,iot23,unsw_nb15,included,0.269843,0.050023,0.084401,0.782933,0.508092


,source_dataset,validation_dataset,precision_attack_difference,recall_attack_difference,f1_attack_difference,accuracy_difference,balanced_accuracy_difference
0,cicids2017,unsw_nb15,-0.003784,-0.000021,-0.000043,-0.000119,-0.000082
1,cicids2017,iot23,-0.000050,-0.000297,-0.000089,0.000688,0.000318
2,unsw_nb15,cicids2017,-0.029110,-0.004374,-0.012942,-0.007253,-0.006173
3,unsw_nb15,iot23,-0.000172,-0.000616,-0.000269,-0.000331,-0.000438
4,iot23,cicids2017,-0.148394,-0.245340,-0.239097,0.000335,-0.091793
5,iot23,unsw_nb15,-0.156969,-0.028727,-0.048569,-0.012149,-0.018366
